In [135]:
import torch
from torch import nn
from torch.optim import Adam
from torchvision import transforms
from torch.utils.data import Dataset,DataLoader
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
import numpy as np
import os

In [136]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [137]:
image_path=[]
labels=[]

path="D://Projects//AI-pipelines//pytorch_exercise//Projects//animal_faces//afhq"
for i in os.listdir(path):
    for label in os.listdir(path+f"//{i}"):
        for img in os.listdir(path+f"//{i}"+f"//{label}"):
            image_path.append(img)
            labels.append(label)

df=pd.DataFrame(zip(image_path,labels),columns=["image_path","label"])


In [138]:
df.head()

,image_path,label
0,flickr_cat_000002.jpg,cat
1,flickr_cat_000003.jpg,cat
2,flickr_cat_000004.jpg,cat
3,flickr_cat_000005.jpg,cat
4,flickr_cat_000006.jpg,cat


In [139]:
df['label'].value_counts()

label
cat     5653
dog     5239
wild    5238
Name: count, dtype: int64

In [140]:
train=df.sample(frac=0.7,random_state=7)
test=df.drop(train.index)

val=test.sample(frac=0.5,random_state=7)
test=test.drop(val.index)

print(train.shape)
print(test.shape)
print(val.shape)

(11291, 2)
(2419, 2)
(2420, 2)


In [141]:
label_encoder=LabelEncoder()

label_encoder.fit(df['label'])

transform=transforms.Compose(
    [
        transforms.Resize((128,128)),
        transforms.ToTensor(),
        transforms.ConvertImageDtype(torch.float)

    ]
)

In [142]:
class CustomImageDataset(Dataset):
    def __init__(self,dataframe,transform=None):
        self.dataframe=dataframe
        self.transform=transform
        self.labels=torch.tensor(label_encoder.transform(dataframe['label'])).to(device)

    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, index):
        img_path=self.dataframe.iloc[index,0]
        label=self.labels[index]

        image=Image.open(img_path).convert('RGB')

        if self.transform:
            image=self.transform(image).to(device)

        return image, label

In [143]:
train_dataset=CustomImageDataset(dataframe=train,transform=transform)
test_dataset=CustomImageDataset(dataframe=test,transform=transform)
val_dataset=CustomImageDataset(dataframe=val,transform=transform)

In [144]:
LR=1e-4
BATCH_SIZE=16
EPOCHS=10

In [145]:
train_loader=DataLoader(train_dataset,shuffle=True,batch_size=BATCH_SIZE)
test_loader=DataLoader(test_dataset,shuffle=True,batch_size=BATCH_SIZE)
val_loader=DataLoader(val_dataset,shuffle=True,batch_size=BATCH_SIZE)

In [146]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1=nn.Conv2d(3,32,kernel_size=3,padding=1) # channel(rgb),No. of kernels,kernel_size
        self.conv2=nn.Conv2d(32,64,kernel_size=3,padding=1)
        self.conv3=nn.Conv2d(64,128,kernel_size=3,padding=1)

        self.pooling=nn.MaxPool2d(2,2)

        self.relu=nn.ReLU()

        self.flatten=nn.Flatten()
        self.linear=nn.Linear((128*16*16),128)

        self.output=nn.Linear(128,len(df['label'].unique()))

    def forward(self,x):
        x=self.conv1(x) #(3,128,128)-->#(32,128,128)
        x=self.pooling(x) #(32,64,64)
        x=self.relu(x) #(32,64,64)

        x=self.conv2(x) #(32,64,64)--> (64,64,64)
        x=self.pooling(x) #(64,32,32)
        x=self.relu(x) #(64,32,32)

        x=self.conv3(x) #(64,32,32)-->(128,32,32)
        x=self.pooling(x) #(128,16,16)
        x=self.relu(x) #(128,16,16)

        x=self.flatten(x)
        x=self.linear(x)
        x=self.output(x)

        return x

In [147]:
model=Net().to(device)

In [148]:
from torchsummary import summary
summary(model,input_size=(3,128,128))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1         [-1, 32, 128, 128]             896
         MaxPool2d-2           [-1, 32, 64, 64]               0
              ReLU-3           [-1, 32, 64, 64]               0
            Conv2d-4           [-1, 64, 64, 64]          18,496
         MaxPool2d-5           [-1, 64, 32, 32]               0
              ReLU-6           [-1, 64, 32, 32]               0
            Conv2d-7          [-1, 128, 32, 32]          73,856
         MaxPool2d-8          [-1, 128, 16, 16]               0
              ReLU-9          [-1, 128, 16, 16]               0
          Flatten-10                [-1, 32768]               0
           Linear-11                  [-1, 128]       4,194,432
           Linear-12                    [-1, 3]             387
Total params: 4,288,067
Trainable params: 4,288,067
Non-trainable params: 0
---------------------------

In [149]:
criterion=nn.CrossEntropyLoss()
optimizer=Adam(model.parameters(),lr=LR)

In [150]:
total_loss_train_plot=[]
total_loss_validation_plot=[]
total_acc_train_plot=[]
total_acc_validation_plot=[]


for epoch in range(EPOCHS):
    total_acc_train=0
    total_loss_train=0
    total_loss_val=0
    total_acc_val=0

    for inputs, labels in train_loader:
        optimizer.zero()
        outputs=model(inputs)

        train_loss=criterion(outputs,labels)
        total_loss_train+=train_loss

        train_loss.backward()

        train_acc=(torch.argmax(outputs,axis=1)==labels).sum().item()

        total_acc_train+=train_acc
        optimizer.step()

    with torch.no_grad():
        for inputs,labels in val_loader:
            outputs=model(inputs)
            val_loss=criterion(outputs,labels)
            total_loss_val+=val_loss

            val_acc=(torch.argmax(outputs,axis=1)==labels).sum().item()

            total_acc_val+=val_acc

    total_loss_train_plot.append(round(total_loss_train/1000, 4))
    total_loss_validation_plot.append(round(total_loss_val/1000, 4))
    total_acc_train_plot.append(round(total_acc_train/(train_dataset.__len__())*100, 4))
    total_acc_validation_plot.append(round(total_acc_val/(val_dataset.__len__())*100, 4))
    print(f'''Epoch {epoch+1}/{EPOCHS}, Train Loss: {round(total_loss_train/100, 4)} Train Accuracy {round((total_acc_train)/train_dataset.__len__() * 100, 4)}
                Validation Loss: {round(total_loss_val/100, 4)} Validation Accuracy: {round((total_acc_val)/val_dataset.__len__() * 100, 4)}''')
    print("="*25)


FileNotFoundError: [Errno 2] No such file or directory: 'flickr_wild_002046.jpg'